In [13]:
import pandas as pd
import numpy as np


In [14]:
air_data = pd.read_csv("CIVE202_Spring2026_MohamedAdam_Project1_RawData.csv.csv")
air_data.head()


,date,monitor_index,humidity,pressure,temperature,voc,analog_input,pm2.5_alt,pm1.0_atm,pm2.5_atm,pm10.0_atm,sensor.latitude,sensor.longitude,sensor.altitude,sensor.name
0,02/23/24,195089,14.377667,912.884333,62.266667,51.998667,0.051333,0.1000,0.000000,0.002500,0.039667,40.050922,-101.533570,3005,Swnphd-Benklemen
1,02/23/24,195365,12.223600,926.403000,71.193400,64.920800,0.000000,0.1800,0.004800,0.020000,0.176000,40.200330,-100.639885,2576,Swnphd-mccook
2,02/23/24,195541,20.095750,905.670750,61.008250,68.307000,0.020000,0.1625,0.004125,0.014812,0.063937,41.128284,-101.720220,3220,Swnphd-ogallala
3,02/24/24,195089,25.368000,911.708833,51.462458,91.176750,0.052667,0.4375,0.099542,0.170667,0.355208,40.050922,-101.533570,3005,Swnphd-Benklemen
4,02/24/24,195365,23.703083,925.282125,56.818208,107.863708,0.000000,0.4750,0.099208,0.231687,0.548583,40.200330,-100.639885,2576,Swnphd-mccook


In [15]:
air_data['Humidity Category'] = np.where(
    air_data['humidity'] < 50, 'Low',
    np.where(air_data['humidity'] <= 80, 'High', 'Very High')
)


In [16]:
air_data['Temperature Category'] = np.select(
    [
        air_data['temperature'] < 32,
        (air_data['temperature'] >= 32) & (air_data['temperature'] <= 50),
        (air_data['temperature'] > 50) & (air_data['temperature'] <= 70),
        air_data['temperature'] > 70
    ],
    ['Below Freezing','Cool','Warm','Hot'],
    default='Unknown'
)


In [17]:
humidity_group = air_data.groupby('Humidity Category').mean(numeric_only=True)

humidity_rank = humidity_group[['voc','pm2.5_atm','pm10.0_atm']].sort_values(
    by='pm2.5_atm', ascending=False
)

humidity_rank


,voc,pm2.5_atm,pm10.0_atm
Humidity Category,,,
Very High,19.395555,533.786399,536.520850
High,252.664069,80.874444,82.561133
Low,279.329347,76.513935,77.880520


In [18]:
temperature_group = air_data.groupby('Temperature Category').mean(numeric_only=True)

temperature_rank = temperature_group[['voc','pm2.5_atm','pm10.0_atm']].sort_values(
    by='pm2.5_atm', ascending=False
)

temperature_rank


,voc,pm2.5_atm,pm10.0_atm
Temperature Category,,,
Unknown,NaN,630.335967,633.339711
Below Freezing,257.628626,273.698560,276.080794
Cool,285.435146,141.658843,143.074475
Warm,257.015827,86.506629,87.428866
Hot,282.145818,18.690352,20.243842


In [19]:
sensor_stats = air_data.groupby('sensor.name').agg({
    'voc': ['mean', 'median'],
    'pm2.5_atm': ['mean', 'median'],
    'pm10.0_atm': ['mean', 'median']
})

sensor_stats.columns = ['VOC Mean','VOC Median',
                        'PM2.5 Mean','PM2.5 Median',
                        'PM10 Mean','PM10 Median']

top5_hotspots = sensor_stats.sort_values(
    by=['PM2.5 Mean','PM10 Mean','VOC Mean'],
    ascending=False
).head(5)

top5_hotspots


,VOC Mean,VOC Median,PM2.5 Mean,PM2.5 Median,PM10 Mean,PM10 Median
sensor.name,,,,,,
Broken Bow,158.285807,138.559646,928.710593,36.050240,929.678512,43.179094
#16 - Richardson County Courthouse,86.758650,84.470500,700.127342,11.977344,701.632446,13.305615
#18 - Southeast District Health Department- Tecumseh,196.150771,138.582583,613.175352,10.322875,614.227248,11.433729
NCDHD O'Neill #11,283.633000,238.404833,164.495078,7.251208,166.132578,8.427437
Swnphd-mccook,353.941581,381.468479,123.011622,4.582281,124.227336,5.372073


In [20]:
max_voc = air_data.loc[air_data['voc'].idxmax()]
max_pm25 = air_data.loc[air_data['pm2.5_atm'].idxmax()]
max_pm10 = air_data.loc[air_data['pm10.0_atm'].idxmax()]

max_voc[['date','sensor.name','voc']]
max_pm25[['date','sensor.name','pm2.5_atm']]
max_pm10[['date','sensor.name','pm10.0_atm']]


date                                     02/18/25
sensor.name    #16 - Richardson County Courthouse
pm10.0_atm                            3784.682542
Name: 7561, dtype: object

In [21]:
pm25_risk_events = air_data[air_data['pm2.5_atm'] >= 35.5]
pm10_risk_events = air_data[air_data['pm10.0_atm'] >= 155]

pm25_summary = pm25_risk_events.groupby('sensor.name').agg(
    Events=('pm2.5_atm','count'),
    Max_PM25=('pm2.5_atm','max')
).sort_values(by='Events', ascending=False)

pm10_summary = pm10_risk_events.groupby('sensor.name').agg(
    Events=('pm10.0_atm','count'),
    Max_PM10=('pm10.0_atm','max')
).sort_values(by='Events', ascending=False)

pm25_summary
pm10_summary


,Events,Max_PM10
sensor.name,,
Broken Bow,176,3206.779708
#16 - Richardson County Courthouse,102,3784.682542
#18 - Southeast District Health Department- Tecumseh,56,2988.441438
Swnphd-mccook,28,3054.484250
NCDHD O'Neill #11,21,2079.294479
TRPHD Dawson Co. Courthouse 25,4,2367.085500
Loup Basin Public Health Department,1,231.757021
FCHD-YPS,1,364.240104
Swnphd-Benklemen,1,188.977500


In [22]:
air_data['Altitude Category'] = pd.cut(
    air_data['sensor.altitude'],
    bins=3,
    labels=['Low Elevation','Medium Elevation','High Elevation']
)

altitude_group = air_data.groupby('Altitude Category').mean(numeric_only=True)

altitude_group[['voc','pm2.5_atm','pm10.0_atm']]


,voc,pm2.5_atm,pm10.0_atm
Altitude Category,,,
Low Elevation,248.479935,105.220386,106.989256
Medium Elevation,281.405971,139.428803,140.745865
High Elevation,309.155679,7.460558,8.597666


In [23]:
pm25_dates = pm25_risk_events[['date','sensor.name','pm2.5_atm']].sort_values('date')
pm10_dates = pm10_risk_events[['date','sensor.name','pm10.0_atm']].sort_values('date')

pm25_dates
pm10_dates


,date,sensor.name,pm10.0_atm
6556,01/01/25,#16 - Richardson County Courthouse,2114.860313
6565,01/01/25,Broken Bow,2107.215604
6576,01/02/25,#16 - Richardson County Courthouse,2136.544792
6585,01/02/25,Broken Bow,1969.514229
6596,01/03/25,#16 - Richardson County Courthouse,2359.675646
...,...,...,...
6505,12/29/24,Broken Bow,1720.299958
6516,12/30/24,#16 - Richardson County Courthouse,1948.080313
6525,12/30/24,Broken Bow,1723.087500
6536,12/31/24,#16 - Richardson County Courthouse,2205.778262


In [24]:
top5_voc = sensor_stats.sort_values(by='VOC Mean', ascending=False).head(5)
top5_pm25 = sensor_stats.sort_values(by='PM2.5 Mean', ascending=False).head(5)
top5_pm10 = sensor_stats.sort_values(by='PM10 Mean', ascending=False).head(5)

top5_voc
top5_pm25
top5_pm10


,VOC Mean,VOC Median,PM2.5 Mean,PM2.5 Median,PM10 Mean,PM10 Median
sensor.name,,,,,,
Broken Bow,158.285807,138.559646,928.710593,36.050240,929.678512,43.179094
#16 - Richardson County Courthouse,86.758650,84.470500,700.127342,11.977344,701.632446,13.305615
#18 - Southeast District Health Department- Tecumseh,196.150771,138.582583,613.175352,10.322875,614.227248,11.433729
NCDHD O'Neill #11,283.633000,238.404833,164.495078,7.251208,166.132578,8.427437
Swnphd-mccook,353.941581,381.468479,123.011622,4.582281,124.227336,5.372073
